# M7 · Capstone: write an evidence-backed optimization PR

**Question:** What is the minimal auditable evidence for a performance claim? Produce a candidate student analysis record from the official dataset, label it `REFERENCE_ANALYSIS_NOT_NEW_GPU_RUN`, and plan a paired A/B with correctness and resource gates. No file is written unless explicitly enabled.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M07_student_challenge/CHALLENGES.md](../course/M07_student_challenge/CHALLENGES.md). Switch language: [中文](./08_capstone_and_pr.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
rows = ns.summary_rows()
b0, f1 = rows["B0"], rows["F1"]
evidence_record = {
    "origin": "committed reference evidence, not a new physical run",
    "status": "REFERENCE_ANALYSIS_NOT_NEW_GPU_RUN",
    "hypothesis": "attention backend may improve same-workload latency",
    "baseline": {"experiment": "B0", "warm_median_s": b0["warm_total_median_s"], "n": b0["n_measure"]},
    "candidate": {"experiment": "F1", "warm_median_s": f1["warm_total_median_s"], "n": f1["n_measure"]},
    "derived": {"speedup_x": round(b0["warm_total_median_s"] / f1["warm_total_median_s"], 4)},
    "source_paths": ["evidence/results/SUMMARY.json", "lab/reference_runs.json"],
    "limitations": ["single W7900-class host", "warm reference population", "student has not run a new A/B"],
    "quality_gate": "requires a separate review for the claimed deployment",
}
import json
print(json.dumps(evidence_record, indent=2, ensure_ascii=False))
assert evidence_record["status"] != "MEASURED_NEW_GPU_RUN"


In [ ]:
WRITE_STUDENT_RECORD = False  # Explicit opt-in only, never evidence/
if WRITE_STUDENT_RECORD:
    path = ns.student_record(evidence_record, "student-reference-analysis", confirmed=True, root=ROOT)
    print("Wrote student-only artifact:", path)
else:
    print("No file written. Set WRITE_STUDENT_RECORD=True to save under runs/notebook-student/.")


## Your turn / Self-check

Propose ONE improvement with a falsifiable hypothesis and matched A/B controls. Include model identity, source SHA, workload, medians, variance, VRAM/RAM, quality status, negative results, and a rerun command. Submit as a draft PR; do not claim the reference calculation is new hardware validation.

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
